---
id: "readme"
template: "markdown"
name: "readme"
position:
  x: 0
  y: -40
dimensions:
  width: 900
  height: 360
config:
  md: |-
    # m_ORDER_LINE_ROUTER (converted from Informatica PowerCenter)

    **Router semantics**: a row goes to *every* group whose condition is TRUE; `DEFAULT1` gets rows that match no group. So this is three independent Filters (not one True/False split), and the default filter is `NOT COALESCE(c1,false) AND NOT COALESCE(c2,false)`.

    | Informatica | Lakeflow Designer |
    |---|---|
    | LKP_PRODUCTS (connected, cached, PRODUCT_ID unique) | `lkp_products` (left join) |
    | EXP_PRICING (IIF / DECODE) | `exp_pricing` |
    | RTR_LINE_TYPE groups HIGH_VALUE / DISCOUNTED / DEFAULT1 | `rtr_high_value`, `rtr_discounted`, `rtr_default` |
    | T_LINES_HIGH_VALUE / T_LINES_DISCOUNTED / T_LINES_STANDARD | 3 outputs |
---

In [0]:
"""
id: "src_order_lines"
template: "source"
templateVersion: "1.0.0"
name: "src_order_lines"
position:
  x: 0
  y: 360
description:
  text: "SQ_ORDER_LINES source."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_order_lines"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_order_lines"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_order_lines.data"] = out["data"]


In [0]:
"""
id: "src_products"
template: "source"
templateVersion: "1.0.0"
name: "src_products"
position:
  x: 0
  y: 515
description:
  text: "LKP_PRODUCTS lookup source (PRODUCTS)."
  hash: ""
previewMode: "1000"
config:
  table_source:
    tableName: "daphnedemos_catalog.migration_skill_tests.src_products"
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "daphnedemos_catalog.migration_skill_tests.src_products"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["src_products.data"] = out["data"]


In [0]:
"""
id: "lkp_products"
template: "join"
templateVersion: "1.0.0"
name: "lkp_products"
position:
  x: 300
  y: 360
description:
  text: "LKP_PRODUCTS: connected lookup = LEFT JOIN (PRODUCT_ID is the key, so 'Use First Value' needs no dedup)."
  hash: ""
previewMode: "1000"
config:
  join_type: "left"
  join_conditions: "left.`product_id` = right.`product_id`"
  expressions:
    - "left.*"
    - "right.`list_price` AS `list_price`"
    - "right.`category` AS `category`"
    - "right.`cost` AS `cost`"
input:
  - node: "src_order_lines"
    input_port: "left"
    output_port: "data"
  - node: "src_products"
    input_port: "right"
    output_port: "data"
"""

# generated from the system
from typing import Dict, Any, List
import pyspark.sql.functions as F

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    join_type = config.get("join_type", "inner").replace(" ", "_")
    join_condition = config.get("join_conditions", "")
    expressions: List[str] = config.get("expressions", [])

    # Alias DataFrames with fixed names so that table-qualified column
    # references in join_conditions and expressions always resolve.
    # The TS serializer emits "left.col" / "right.col" regardless of the
    # actual table names, so these aliases must match exactly.
    df_left = inputs.get("left")
    df_right = inputs.get("right")
    if df_left is None or df_right is None:
        raise ValueError("Both left and right inputs must be connected")
    df_left = df_left.alias("left")
    df_right = df_right.alias("right")

    # join_condition is a plain SQL expression string serialized by the TS
    # side (same pattern as filter's condition field).
    if not join_condition:
        result = df_left.join(df_right, how=join_type)
    else:
        result = df_left.join(df_right, F.expr(join_condition), how=join_type)

    # Apply optional SELECT expressions
    if expressions:
        result = result.selectExpr(*expressions)

    return {"joined_data": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "join_type": "left",
    "join_conditions": "left.`product_id` = right.`product_id`",
    "expressions": [
        "left.*",
        "right.`list_price` AS `list_price`",
        "right.`category` AS `category`",
        "right.`cost` AS `cost`"
    ]
}
inputs = {
    "left": ctx["src_order_lines.data"],
    "right": ctx["src_products.data"]
}
out = run(config, inputs, spark)
ctx["lkp_products.joined_data"] = out["joined_data"]


In [0]:
"""
id: "exp_pricing"
template: "transform"
templateVersion: "1.0.0"
name: "exp_pricing"
position:
  x: 600
  y: 360
description:
  text: "EXP_PRICING: IIF(ISNULL(..)) -> COALESCE; DECODE -> CASE."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "*"
    - "COALESCE(unit_price, list_price) AS price"
    - "ROUND(quantity * COALESCE(unit_price, list_price) * (1 - discount_pct), 2) AS net_amount"
    - "ROUND(quantity * COALESCE(unit_price, list_price) * (1 - discount_pct), 2) - ROUND(quantity * cost, 2) AS margin"
    - "CASE category WHEN 'Dairy' THEN 'FRESH' WHEN 'Bakery' THEN 'FRESH' WHEN 'Produce' THEN 'FRESH' WHEN 'Electronics' THEN 'NON_FOOD' WHEN 'Household' THEN 'NON_FOOD' ELSE 'DRY' END AS category_group"
input:
  - node: "lkp_products"
    input_port: "data"
    output_port: "joined_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "*",
        "COALESCE(unit_price, list_price) AS price",
        "ROUND(quantity * COALESCE(unit_price, list_price) * (1 - discount_pct), 2) AS net_amount",
        "ROUND(quantity * COALESCE(unit_price, list_price) * (1 - discount_pct), 2) - ROUND(quantity * cost, 2) AS margin",
        "CASE category WHEN 'Dairy' THEN 'FRESH' WHEN 'Bakery' THEN 'FRESH' WHEN 'Produce' THEN 'FRESH' WHEN 'Electronics' THEN 'NON_FOOD' WHEN 'Household' THEN 'NON_FOOD' ELSE 'DRY' END AS category_group"
    ]
}
inputs = {
    "data": ctx["lkp_products.joined_data"]
}
out = run(config, inputs, spark)
ctx["exp_pricing.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "rtr_high_value"
template: "filter"
templateVersion: "1.0.0"
name: "rtr_high_value"
position:
  x: 900
  y: 360
description:
  text: "RTR_LINE_TYPE group HIGH_VALUE."
  hash: ""
previewMode: "1000"
config:
  condition: "net_amount >= 300"
input:
  - node: "exp_pricing"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "net_amount >= 300"
}
inputs = {
    "data": ctx["exp_pricing.transformed_data"]
}
out = run(config, inputs, spark)
ctx["rtr_high_value.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "rtr_discounted"
template: "filter"
templateVersion: "1.0.0"
name: "rtr_discounted"
position:
  x: 900
  y: 515
description:
  text: "RTR_LINE_TYPE group DISCOUNTED (rows can also be in HIGH_VALUE)."
  hash: ""
previewMode: "1000"
config:
  condition: "discount_pct >= 0.25"
input:
  - node: "exp_pricing"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "discount_pct >= 0.25"
}
inputs = {
    "data": ctx["exp_pricing.transformed_data"]
}
out = run(config, inputs, spark)
ctx["rtr_discounted.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "rtr_default"
template: "filter"
templateVersion: "1.0.0"
name: "rtr_default"
position:
  x: 900
  y: 670
description:
  text: "RTR_LINE_TYPE DEFAULT1: rows matching no group."
  hash: ""
previewMode: "1000"
config:
  condition: "NOT COALESCE(net_amount >= 300, false) AND NOT COALESCE(discount_pct >= 0.25, false)"
input:
  - node: "exp_pricing"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    condition = config.get("condition", "")

    if not condition:
        return {"filtered_data": df}

    return {"filtered_data": df.filter(condition)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "condition": "NOT COALESCE(net_amount >= 300, false) AND NOT COALESCE(discount_pct >= 0.25, false)"
}
inputs = {
    "data": ctx["exp_pricing.transformed_data"]
}
out = run(config, inputs, spark)
ctx["rtr_default.filtered_data"] = out["filtered_data"]


In [0]:
"""
id: "map_high_value"
template: "transform"
templateVersion: "1.0.0"
name: "map_high_value"
position:
  x: 1200
  y: 360
description:
  text: "Ports -> T_LINES_HIGH_VALUE."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "order_id"
    - "line_no"
    - "product_id"
    - "category"
    - "category_group"
    - "quantity"
    - "price"
    - "discount_pct"
    - "net_amount"
    - "margin"
input:
  - node: "rtr_high_value"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_id",
        "line_no",
        "product_id",
        "category",
        "category_group",
        "quantity",
        "price",
        "discount_pct",
        "net_amount",
        "margin"
    ]
}
inputs = {
    "data": ctx["rtr_high_value.filtered_data"]
}
out = run(config, inputs, spark)
ctx["map_high_value.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "map_discounted"
template: "transform"
templateVersion: "1.0.0"
name: "map_discounted"
position:
  x: 1200
  y: 515
description:
  text: "Ports -> T_LINES_DISCOUNTED."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "order_id"
    - "line_no"
    - "product_id"
    - "category"
    - "category_group"
    - "quantity"
    - "price"
    - "discount_pct"
    - "net_amount"
    - "margin"
input:
  - node: "rtr_discounted"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_id",
        "line_no",
        "product_id",
        "category",
        "category_group",
        "quantity",
        "price",
        "discount_pct",
        "net_amount",
        "margin"
    ]
}
inputs = {
    "data": ctx["rtr_discounted.filtered_data"]
}
out = run(config, inputs, spark)
ctx["map_discounted.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "map_standard"
template: "transform"
templateVersion: "1.0.0"
name: "map_standard"
position:
  x: 1200
  y: 670
description:
  text: "Ports -> T_LINES_STANDARD."
  hash: ""
previewMode: "1000"
config:
  expressions:
    - "order_id"
    - "line_no"
    - "product_id"
    - "category"
    - "category_group"
    - "quantity"
    - "price"
    - "discount_pct"
    - "net_amount"
    - "margin"
input:
  - node: "rtr_default"
    input_port: "data"
    output_port: "filtered_data"
"""

# generated from the system
from typing import Dict, Any, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    expressions: List[str] = config.get("expressions", [])

    if not expressions:
        return {"transformed_data": df}

    return {"transformed_data": df.selectExpr(*expressions)}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "expressions": [
        "order_id",
        "line_no",
        "product_id",
        "category",
        "category_group",
        "quantity",
        "price",
        "discount_pct",
        "net_amount",
        "margin"
    ]
}
inputs = {
    "data": ctx["rtr_default.filtered_data"]
}
out = run(config, inputs, spark)
ctx["map_standard.transformed_data"] = out["transformed_data"]


In [0]:
"""
id: "out_lines_high_value"
template: "output"
templateVersion: "1.0.0"
name: "out_lines_high_value"
position:
  x: 1500
  y: 360
description:
  text: "T_LINES_HIGH_VALUE"
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_03_lines_high_value"
input:
  - node: "map_high_value"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_03_lines_high_value"
}
inputs = {
    "data": ctx["map_high_value.transformed_data"]
}
out = run(config, inputs, spark)


In [0]:
"""
id: "out_lines_discounted"
template: "output"
templateVersion: "1.0.0"
name: "out_lines_discounted"
position:
  x: 1500
  y: 515
description:
  text: "T_LINES_DISCOUNTED"
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_03_lines_discounted"
input:
  - node: "map_discounted"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_03_lines_discounted"
}
inputs = {
    "data": ctx["map_discounted.transformed_data"]
}
out = run(config, inputs, spark)


In [0]:
"""
id: "out_lines_standard"
template: "output"
templateVersion: "1.0.0"
name: "out_lines_standard"
position:
  x: 1500
  y: 670
description:
  text: "T_LINES_STANDARD"
  hash: ""
previewMode: "full"
config:
  catalog: "daphnedemos_catalog"
  schema: "migration_skill_tests"
  table_name: "infa_03_lines_standard"
input:
  - node: "map_standard"
    input_port: "data"
    output_port: "transformed_data"
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "daphnedemos_catalog",
    "schema": "migration_skill_tests",
    "table_name": "infa_03_lines_standard"
}
inputs = {
    "data": ctx["map_standard.transformed_data"]
}
out = run(config, inputs, spark)
